In [1]:
import pandas as pd
import numpy as np

In [3]:
# NETTOYAGE
# Entrée  : le fichier DVF brut (1 ligne = 1 parcelle/local impliqué
#           dans une vente, PAS 1 ligne = 1 bien vendu).
# Sorties : - dvf_residentiel_clean.csv  -> pour le clustering (Maison / Appartement / Appartement VEFA)
#           - dvf_terrains_locaux_clean.csv -> terrains nus et locaux commerciaux, mis de côté (logique de prix différente)

RAW_PATH = "ValeursFoncieres-2025.txt"

df = pd.read_csv(RAW_PATH, sep="|", dtype=str, encoding="utf-8",)
print("Lignes brutes :", len(df))

# 1. SUPPRESSION DES COLONNES INUTILES
# pour la segmentation marché (localisation / type / prix)
cols_to_drop = [
    "Identifiant de document", "Reference document",
    "1 Articles CGI", "2 Articles CGI", "3 Articles CGI",
    "4 Articles CGI", "5 Articles CGI",
    "No Volume",
    "1er lot", "2eme lot", "3eme lot", "4eme lot", "5eme lot",
    "Surface Carrez du 2eme lot", "Surface Carrez du 3eme lot",
    "Surface Carrez du 4eme lot", "Surface Carrez du 5eme lot",
    "Identifiant local",
    "Prefixe de section",
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

Lignes brutes : 3714829


In [4]:
# 2. SUPPRESSION DES DOUBLONS EXACTS
# les lignes identiques à cause de la jointure parcelles x lots
before = len(df)
df = df.drop_duplicates()
print(f"Doublons exacts supprimés : {before - len(df)}")
print("Lignes restantes :", len(df))

Doublons exacts supprimés : 454335
Lignes restantes : 3260494


In [5]:
# CONVERSION DES TYPES
def to_float(serie):
    return pd.to_numeric(
        serie.str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce",
    )
 
df["Valeur fonciere"] = to_float(df["Valeur fonciere"])
df["Surface reelle bati"] = to_float(df["Surface reelle bati"])
df["Surface terrain"] = to_float(df["Surface terrain"])
df["Surface Carrez du 1er lot"] = to_float(df["Surface Carrez du 1er lot"])
df["Nombre pieces principales"] = to_float(df["Nombre pieces principales"])
df["Nombre de lots"] = to_float(df["Nombre de lots"])
df["Date mutation"] = pd.to_datetime(df["Date mutation"], format="%d/%m/%Y", errors="coerce")

# 3. RECONSTITUTION D'UNE MUTATION (= UNE VENTE)
# Le fichier DVF liste UNE LIGNE PAR PARCELLE/LOCAL impliqué dans la vente, pas une ligne par bien vendu. Une maison + sa dépendance + 2 parcelles de terrain donnent 4 lignes pour UNE SEULE transaction.
# Du coup on va regrouper : département + commune + no disposition + date + valeur (ces 5 champs sont identiques pour toutes les lignes d'une même mutation).
id_cols = ["Code departement", "Code commune", "No disposition", "Date mutation", "Valeur fonciere"]

df["id_mutation"] = (df[id_cols].fillna("NA").astype(str).apply(lambda row: "_".join(row.values), axis=1))

print("Nombre de mutations distinctes :", df["id_mutation"].nunique())

Nombre de mutations distinctes : 1329258


In [ ]:
# 4. ADRESSE COMPLETE (avant agrégation, car peut varier légèrement entre les lignes d'une même mutation -> on prendra la 1ère adresse non vide au moment du groupby)
def build_adresse(row):
    parts = [row.get("No voie"), row.get("B/T/Q"), row.get("Type de voie"), row.get("Voie")]
    parts = [str(p) for p in parts if pd.notna(p) and str(p).strip() != ""]
    return " ".join(parts).strip()
 
df["adresse"] = df.apply(build_adresse, axis=1)

In [ ]:
# 5. HARMONISATION DU TYPE DE BIEN AU NIVEAU MUTATION
# le "type de bien" d'une mutation est déterminé par la présence d'un local d'habitation/commercial.
# Une mutation sans aucun "Type local" renseigné = vente de terrain nu.
def classify_mutation(types_locaux):
    types = set(t for t in types_locaux if pd.notna(t) and t != "")
    has_maison = "Maison" in types
    has_appart = "Appartement" in types
    has_local_pro = "Local industriel. commercial ou assimilé" in types
    n_types = sum([has_maison, has_appart, has_local_pro])
 
    if n_types == 0:
        return "Terrain"
    if n_types > 1:
        return "Mixte"          # ex : maison + local commercial dans la même vente
    if has_maison:
        return "Maison"
    if has_appart:
        return "Appartement"
    return "Local commercial"
 
base = df.groupby("id_mutation").agg(
    date_mutation=("Date mutation", "first"),
    nature_mutation=("Nature mutation", "first"),
    valeur_fonciere=("Valeur fonciere", "first"),
    code_postal=("Code postal", "first"),
    commune=("Commune", "first"),
    code_departement=("Code departement", "first"),
    code_commune=("Code commune", "first"),
    adresse=("adresse", lambda s: next((x for x in s if x), "")),
    nb_lignes_brutes=("Type local", "size"),
).reset_index()
 
# "Locaux" (bâti) : quand une maison/appartement est implanté sur plusieurs subdivisions cadastrales (nature de culture différente sur une même parcelle, ex: "S" et "AG"), DVF REPETE la ligne du local à l'identique (même surface, même nb de pièces), une fois par subdivision.
# Ce n'est PAS un doublon strict (les colonnes "Nature culture"/"Surface terrain" diffèrent) donc drop_duplicates() ne doit pas le supprimé.
# Si on somme brut, on compte la même maison 2 ou 3 fois. On déduplique donc sur la signature du local (type + surface + pièces) avant de sommer.
locaux = df.dropna(subset=["Type local"]).drop_duplicates(subset=["id_mutation", "Type local", "Surface reelle bati", "Nombre pieces principales"])
 
def agg_locaux(g):
    types = set(g["Type local"])
    habitation_mask = g["Type local"].isin(["Maison", "Appartement", "Local industriel. commercial ou assimilé"])
    habitation_rows = g[habitation_mask]
    # Nombre de biens d'habitation DISTINCTS dans la mutation (après dédup des répétitions d'un même bien sur plusieurs subdivisions cadastrales, déjà faite en amont).
    # Si > 1, il s'agit très probablement d'une vente en bloc / lotissement (plusieurs maisons/appartements différents vendus dans une même transaction) et non d'un bien unique -> à traiter à part.
    nb_locaux_habitation = habitation_rows[["Type local", "Surface reelle bati", "Nombre pieces principales"]].drop_duplicates().shape[0]
    return pd.Series({
        "type_bien": classify_mutation(types),
        "surface_bati": habitation_rows["Surface reelle bati"].sum(min_count=1),
        "surface_carrez": habitation_rows["Surface Carrez du 1er lot"].sum(min_count=1),
        "nb_pieces": habitation_rows["Nombre pieces principales"].max(),
        "nb_dependances": (g["Type local"] == "Dépendance").sum(),
        "nb_locaux_habitation": nb_locaux_habitation,
    })
 
bati_agg = locaux.groupby("id_mutation").apply(agg_locaux, include_groups=False).reset_index()
 
# Correctif VEFA : les logements vendus "en l'état futur d'achèvement" n'ont souvent AUCUN "Type local" renseigné (le bien n'existe pas encore au cadastre) alors qu'une surface carré est bien indiquée.
# Sans ce correctif, ces ventes tombent à tort dans "Terrain" alors qu'il s'agit presque toujours d'appartements neufs.
vefa_carrez = (df[df["Nature mutation"] == "Vente en l'état futur d'achèvement"].groupby("id_mutation")["Surface Carrez du 1er lot"].sum())
vefa_mutations_sans_type = set(vefa_carrez[vefa_carrez > 0].index) - set(bati_agg["id_mutation"])
vefa_fix = pd.DataFrame({
    "id_mutation": list(vefa_mutations_sans_type),
    "type_bien": "Appartement (VEFA)",
    "surface_bati": [vefa_carrez[m] for m in vefa_mutations_sans_type],
    "surface_carrez": [vefa_carrez[m] for m in vefa_mutations_sans_type],
    "nb_pieces": np.nan,
    "nb_dependances": 0,
})
bati_agg = pd.concat([bati_agg, vefa_fix], ignore_index=True)
 
# Terrain : une parcelle peut elle aussi apparaître sur plusieurs lignes (une fois par local qui s'y trouve).
# On déduplique sur l'identité de la parcelle (section + no plan + nature de culture) avant de sommer les surfaces, sinon même problème de double-comptage.
parcelles = df.drop_duplicates(subset=["id_mutation", "Section", "No plan", "Nature culture", "Surface terrain"])
terrain_agg = (parcelles.groupby("id_mutation")["Surface terrain"].sum().reset_index(name="surface_terrain_totale"))
agg = base.merge(bati_agg, on="id_mutation", how="left").merge(terrain_agg, on="id_mutation", how="left")

# Mutations sans aucun local (= vente de terrain pur) : type_bien manquant
agg["type_bien"] = agg["type_bien"].fillna("Terrain")
agg["surface_bati"] = agg["surface_bati"].fillna(0)
agg["nb_dependances"] = agg["nb_dependances"].fillna(0)
 
print("\nRépartition des types de biens (mutations reconstituées) :")
print(agg["type_bien"].value_counts())
print("\nExemple de lignes :")
print(agg.head(10).to_string())


Répartition des types de biens (mutations reconstituées) :
type_bien
Maison                473734
Terrain               392622
Appartement           380682
Local commercial       50188
Mixte                  25358
Appartement (VEFA)      6674
Name: count, dtype: int64

Exemple de lignes :
                         id_mutation date_mutation nature_mutation  valeur_fonciere code_postal             commune code_departement code_commune                 adresse  nb_lignes_brutes type_bien  surface_bati  surface_carrez  nb_pieces  nb_dependances  surface_terrain_totale
0   01_100_000001_2025-02-07_10000.0    2025-02-07           Vente          10000.0        1510  CHEIGNIEU-LA-BALME               01          100                LA BALME                 2   Terrain           0.0             NaN        NaN             0.0                  2958.0
1  01_100_000001_2025-02-18_156000.0    2025-02-18           Vente         156000.0        1510  CHEIGNIEU-LA-BALME               01          100      

In [ ]:
# 6. FILTRAGE DES VALEURS ABERRANTES
n0 = len(agg)
 
# On garde uniquement les vraies ventes (on écarte les mutations à titre gratuit mal classées, adjudications, etc. si présentes)
agg = agg[agg["nature_mutation"].isin(["Vente", "Vente en l'état futur d'achèvement"])]
 
# Prix manifestement symboliques ou nuls (donations déguisées, ventes entre proches, valeur non renseignée) : on utilise un seuil bas car un vrai bien immobilier ne se vend jamais à 1€
# ou quelques centaines d'euros (on a vu 1,00 / 100,00 / 700,00 dans l'échantillon -> clairement pas des prix de marché sauf s'il s'agit de terrains agricoles nus, cf. point (d))
agg = agg[agg["valeur_fonciere"].notna() & (agg["valeur_fonciere"] > 0)]
 
# Pour le cœur de la segmentation "marché résidentiel", on se concentre sur les biens avec du bâti (Maison/Appartement/VEFA).
# Les mutations "Terrain" et "Local commercial" sont conservées à part car elles répondent à une autre logique de valorisation (prix au m² de terrain nu, zonage, etc.)
residentiel = agg[agg["type_bien"].isin(["Maison", "Appartement", "Appartement (VEFA)"])].copy()
 
# Prix au m² : seul indicateur vraiment comparable pour le clustering. On exige une surface bâtie > 0 pour le calculer.
residentiel = residentiel[residentiel["surface_bati"] > 0]
residentiel["prix_m2"] = residentiel["valeur_fonciere"] / residentiel["surface_bati"]
 
# Valeurs aberrantes de prix/m² : on utilise l'IQR PAR DÉPARTEMENT (un prix/m² normal à Paris est aberrant en campagne et inversement, un seuil global n'a pas de sens sur un dataset national).
def filter_iqr_vectorized(df_in, col="prix_m2", group_col="code_departement", k=1.5, min_size=10):
    g = df_in.groupby(group_col)[col]
    size = g.transform("size")
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    keep = (size < min_size) | df_in[col].between(low, high)
    return df_in[keep]
 
residentiel = filter_iqr_vectorized(residentiel)
 
print(f"\nMutations résidentielles avant filtrage prix/m² aberrant : {n0}")
print(f"Mutations résidentielles conservées après nettoyage complet : {len(residentiel)}")
print("\nAperçu du dataset final (segmentation) :")
cols_final = ["id_mutation", "date_mutation", "commune", "code_postal", "type_bien", "valeur_fonciere", "surface_bati", "nb_pieces", "prix_m2"]
print(residentiel[cols_final].sort_values("prix_m2", ascending=False).head(15).to_string())


Mutations résidentielles avant filtrage prix/m² aberrant : 1329258
Mutations résidentielles conservées après nettoyage complet : 828056

Aperçu du dataset final (segmentation) :
                                id_mutation date_mutation   commune code_postal    type_bien  valeur_fonciere  surface_bati  nb_pieces       prix_m2
973020   75_101_000001_2025-10-31_1091592.0    2025-10-31  PARIS 01       75001  Appartement        1091592.0          65.0        2.0  16793.723077
1010311  75_119_000001_2025-10-31_2015100.0    2025-10-31  PARIS 19       75019       Maison        2015100.0         120.0        5.0  16792.500000
981981    75_109_000001_2025-12-05_403000.0    2025-12-05  PARIS 09       75009  Appartement         403000.0          24.0        1.0  16791.666667
975058   75_104_000001_2025-06-13_1125000.0    2025-06-13  PARIS 04       75004  Appartement        1125000.0          67.0        4.0  16791.044776
975539   75_105_000001_2025-01-27_1057281.0    2025-01-27  PARIS 05       75

In [9]:
# 7. EXPORT
residentiel.to_csv("dvf_residentiel_clean.csv", index=False)
agg[agg["type_bien"].isin(["Terrain", "Local commercial"])].to_csv(
    "dvf_terrains_locaux_clean.csv", index=False
)
print("\nFichiers exportés : dvf_residentiel_clean.csv / dvf_terrains_locaux_clean.csv")


Fichiers exportés : dvf_residentiel_clean.csv / dvf_terrains_locaux_clean.csv
